# Case study: who cancels? Explaining a hotel booking model
**Advanced Predictive Analytics and Data Mining 2026-27, Week 8, Explainable AI: hands-on**

A hotel group wants to know which bookings will be cancelled, so that it can overbook with care and contact the right guests.
We train an accurate black box (XGBoost), and then we do what Week 7 was about: we ask the model **why**.

The answer is not the one a hotel manager expects, and finding out what is going on is the real exercise.

| Step | Question |
|---|---|
| 1 | How good is the model? |
| 2 | What does the model use, and in which direction? |
| 3 | Is the model wrong, or do the data say the same? |
| 4 | Who are these bookings? |
| 5 | Can we use the explanation to decide what to do? |
| 6 | Does the model need its most important column? |
| 7 | Is any column too good to be true? |
| 8 | Why this booking? |

**Data.** 119,210 bookings of two hotels in Portugal (a resort in the Algarve and a city hotel in Lisbon), arrivals from July 2015 to August 2017.
Source: Antonio, de Almeida and Nunes, *Hotel booking demand datasets*, Data in Brief 22 (2019), in the cleaned version of TidyTuesday and DataCamp (one-hot encoded, CC BY 4.0).

Each step below is one step of the interactive page [Who cancels?](https://kalampokis.github.io/dm/Week8/HotelCancellations.html), in the same order and with the same numbers.

**Where to run it.** Anywhere: a laptop, Google Colab, or a notebook on Google Cloud (Colab Enterprise, Vertex AI Workbench).
The notebook downloads the data itself (14 MB), so there is nothing to upload. It runs in about two minutes on two CPU cores. No GPU is needed.

Checked with xgboost 3.4.1, shap 0.54.1, scikit-learn 1.9.1, pandas 3.0 and numpy 2.5. With other versions the last decimal of some numbers may differ.

## Setup
`shap` is not installed in Colab by default. The first cell installs it if it is missing.

In [ ]:
try:
    import shap
except ImportError:
    %pip install -q shap
    import shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

print("xgboost", xgb.__version__, "| shap", shap.__version__, "| pandas", pd.__version__)

## Step 1. How good is the model?
One row is one booking. `is_canceled` is the target. All the other columns are numbers: counts, prices, and 0/1 columns
that came out of one-hot encoding (for example the three `deposit_type_...` columns).

In [ ]:
DATA = "https://kalampokis.github.io/dm/Week8/data/hotel_bookings_clean.csv"   # or the path of a local copy
bookings = pd.read_csv(DATA)

print(bookings.shape)                                  # (119210, 53)
print(round(bookings["is_canceled"].mean(), 3))        # 0.371: more than one booking in three is cancelled

In [ ]:
X = bookings.drop(columns="is_canceled")
y = bookings["is_canceled"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=123)

# The settings that last year's random search selected. One model, trained in about two seconds.
PARAMS = dict(n_estimators=100, max_depth=11, learning_rate=0.2278, reg_alpha=0.001, random_state=123)
model = xgb.XGBClassifier(**PARAMS).fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]   # predicted probability of cancellation
pred = proba > 0.5
print("test AUC :", round(roc_auc_score(y_test, proba), 3))                              # 0.918
print("train AUC:", round(roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]), 3))   # 0.982: deep trees memorise part of the training data
print("accuracy :", round(accuracy_score(y_test, pred), 3))                               # 0.851

Accuracy alone hides the bookings we care about. Read the confusion matrix for the cancellations:
**recall** = of the bookings that were cancelled, how many did the model flag? **precision** = of the bookings it flagged, how many were cancelled?

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(pd.DataFrame([[tn, fp], [fn, tp]], index=["stayed", "cancelled"], columns=["predicted: stays", "predicted: cancels"]))
print("recall   :", round(tp / (tp + fn), 3))   # 0.743 = 10802 of 14536
print("precision:", round(tp / (tp + fp), 3))   # 0.836 = 10802 of 12915

### Optional: tune the model yourself
The search below tries 30 random settings with 5-fold cross-validation, 150 models in total. It takes about two minutes on two CPU cores,
and it is the only part of the notebook that gets faster with more cores. Set `RUN_SEARCH = True` to run it.
On our run it ended at a cross-validated AUC of 0.914, no better than the fixed settings above.

In [ ]:
RUN_SEARCH = False

if RUN_SEARCH:
    from sklearn.model_selection import RandomizedSearchCV
    grid = {"max_depth": list(range(3, 12)),
            "reg_alpha": [0, 0.001, 0.01, 0.1],
            "subsample": [0.25, 0.5, 0.75, 1],
            "learning_rate": np.linspace(0.01, 0.5, 10),
            "n_estimators": [50, 100, 200]}
    search = RandomizedSearchCV(xgb.XGBClassifier(random_state=123), grid, n_iter=30, cv=5,
                                scoring="roc_auc", n_jobs=-1, random_state=123, verbose=1)
    search.fit(X_train, y_train)
    print("best settings:", search.best_params_)
    print("best cross-validated AUC:", round(search.best_score_, 4))

## Step 2. What does the model use, and in which direction?
### The importance that XGBoost reports
`gain` = how much the splits on a column improved the model, as a share of the total.

In [ ]:
gain = pd.Series(model.get_booster().get_score(importance_type="gain"))
gain = (gain / gain.sum()).sort_values(ascending=False)
print(gain.head(5).round(3))   # deposit_type_Non_Refund 0.781, required_car_parking_spaces 0.065

gain.head(10)[::-1].plot.barh(figsize=(7, 4), color="#4F6FE8")
plt.xlabel("share of the total gain"); plt.tight_layout(); plt.show()

One column takes almost four fifths of the gain: whether the booking has a **non-refundable deposit**.

**Stop and predict.** A guest who has paid the whole stay in advance and cannot get the money back: is this guest more likely or less likely to cancel?
Write your answer down before you run the next cells. Gain cannot tell you: it has no direction.

### SHAP: importance with a direction
`TreeExplainer` is the SHAP algorithm for tree models. It reads the trees directly, so it is exact for the model and takes seconds.
(A model-agnostic explainer such as `shap.Explainer(model.predict_proba, X_train)` treats the model as a black box and needs minutes for the same job.)

`model_output="probability"` gives contributions in probability, which we multiply by 100 to read them as **percentage points**.

In [ ]:
background = X_train.sample(100, random_state=0)   # the "average booking" the contributions are measured against
sample = X_test.sample(2000, random_state=0)       # the bookings we explain

explainer = shap.TreeExplainer(model, data=background, model_output="probability")
sv = explainer(sample)                             # about 20 seconds
sv = shap.Explanation(values=sv.values * 100, base_values=sv.base_values * 100,
                      data=sv.data, feature_names=sv.feature_names)

print("average prediction on the background:", round(float(sv.base_values[0]), 1))   # 41.2

In [ ]:
contrib = pd.DataFrame(sv.values, columns=X.columns, index=sample.index)   # one row per booking, one column per feature
print(contrib.abs().mean().sort_values(ascending=False).head(5).round(1))  # deposit_type_Non_Refund 10.8, total_of_special_requests 7.6, lead_time 6.1

shap.plots.bar(sv, max_display=10)

In [ ]:
shap.plots.beeswarm(sv, max_display=10)

Read the first row of the beeswarm. Red dots are bookings **with** a non-refundable deposit, and they are far to the **right**:
the deposit pushes the prediction towards *cancel*. In numbers:

In [ ]:
has_nr = sample["deposit_type_Non_Refund"] == 1
print("bookings with a non-refundable deposit in the sample:", has_nr.sum())               # 239
print("their average contribution :", round(contrib.loc[has_nr, "deposit_type_Non_Refund"].mean(), 1))    # +38.9 points
print("everybody else             :", round(contrib.loc[~has_nr, "deposit_type_Non_Refund"].mean(), 1))   # -7.0 points

### One question, three columns
One-hot encoding split the deposit into three 0/1 columns, and SHAP shares the credit among them.
Shapley values add up, so the contribution of *the deposit* is the sum of the three.

In [ ]:
deposit_cols = ["deposit_type_No_Deposit", "deposit_type_Non_Refund", "deposit_type_Refundable"]
deposit_contrib = contrib[deposit_cols].sum(axis=1)
print("non-refundable:", round(deposit_contrib[has_nr].mean(), 1))    # +43.8 points
print("everybody else:", round(deposit_contrib[~has_nr].mean(), 1))   # -7.8 points

## Step 3. Is the model wrong, or do the data say the same?
An explanation describes the **model**. Before believing it or blaming it, look at the data the model learned from.

In [ ]:
bookings["deposit"] = bookings[deposit_cols].idxmax(axis=1).str.replace("deposit_type_", "")
by_deposit = bookings.groupby("deposit")["is_canceled"].agg(bookings="size", cancelled="sum", rate="mean")
print(by_deposit.round(3))   # Non_Refund: 14494 of 14587 cancelled = 0.994. No_Deposit: 0.284

The model is faithful to its data: 14,494 of the 14,587 bookings with a non-refundable deposit were cancelled.
Only 93 guests arrived. So the question moves from the model to the data: **who are these bookings?**

## Step 4. Who are these bookings?
Compare the bookings with a non-refundable deposit to all the others.

In [ ]:
nr = bookings["deposit"] == "Non_Refund"
describe = {
    "group booking (market segment)":      lambda d: d["market_segment_Groups"].mean(),
    "offline travel agent, tour operator": lambda d: d["market_segment_Offline_TA_TO"].mean(),
    "online travel agent":                 lambda d: d["market_segment_Online_TA"].mean(),
    "no special request at all":           lambda d: (d["total_of_special_requests"] == 0).mean(),
    "asked for a parking space":           lambda d: (d["required_car_parking_spaces"] > 0).mean(),
    "travelling with children":            lambda d: (d["children"] > 0).mean(),
    "has cancelled before":                lambda d: (d["previous_cancellations"] > 0).mean(),
    "city hotel":                          lambda d: d["hotel_City"].mean(),
    "room type A":                         lambda d: d["reserved_room_type_A"].mean(),
    "median lead time (days)":             lambda d: d["lead_time"].median(),
}
profile = pd.DataFrame({"non-refundable": {k: f(bookings[nr]) for k, f in describe.items()},
                        "all the others": {k: f(bookings[~nr]) for k, f in describe.items()}})
print(profile.round(3))   # group booking 0.629 against 0.101; no special request 0.998 against 0.532; lead time 183 against 57

These are not families who paid for their holiday and changed their mind. They are bookings made in bulk, half a year ahead,
through groups and tour operators, for the standard room, with no request of any kind.

The data cannot tell us *why* they were cancelled. Some hypotheses to take to the hotel:
- blocks of rooms held by tour operators and released when they were not sold
- bookings made only to support a visa application, with a card that could not be charged (the authors of the dataset report this for other hotels of their study)
- the way the column was built: the dataset derives the deposit type from the payments recorded in the hotel system, it is not a field the guest filled in

Explainable AI pointed at the place to look. Deciding what happened there needs a person who knows the hotel.

## Step 5. Can we use the explanation to decide what to do?
A manager reads "the deposit is the most important driver of cancellations" and wants to act. Ask the model: take the test bookings
**without** a deposit and change only the deposit to non-refundable.

In [ ]:
no_deposit = X_test[X_test["deposit_type_No_Deposit"] == 1]
with_deposit = no_deposit.assign(deposit_type_No_Deposit=0, deposit_type_Non_Refund=1)

print("bookings changed:", len(no_deposit))                                                                  # 34506
print("average prediction as they are    :", round(model.predict_proba(no_deposit)[:, 1].mean(), 3))        # 0.282
print("with a non-refundable deposit     :", round(model.predict_proba(with_deposit)[:, 1].mean(), 3))      # 0.913

According to the model, asking these guests for a non-refundable deposit would raise their cancellations from 28.2% to 91.3%.
Nobody believes that.

The model learned that the deposit **goes together with** cancellation in this hotel's history. It did not learn what happens when the hotel **changes** the deposit policy.
A what-if on the model answers "what would the model say?", never "what would the guest do?". For the second question you need an experiment, for example an A/B test of two deposit policies.

## Step 6. Does the model need its most important column?
The column takes 78% of the gain. Guess what happens to the test AUC if we delete the three deposit columns and train again. Then run the cell.

In [ ]:
def test_auc(columns):
    m = xgb.XGBClassifier(**PARAMS).fit(X_train[columns], y_train)
    return m, roc_auc_score(y_test, m.predict_proba(X_test[columns])[:, 1])

no_deposit_cols = [c for c in X.columns if c not in deposit_cols]
model_no_deposit, auc_no_deposit = test_auc(no_deposit_cols)
print("with the deposit columns   :", round(roc_auc_score(y_test, proba), 3))   # 0.918
print("without the deposit columns:", round(auc_no_deposit, 3))                  # 0.917

Almost nothing is lost. The model does not need to be told about the deposit, because the other columns give it away.
Check it: train a second model whose only job is to **guess** the deposit column from the rest.

In [ ]:
target_train = X_train["deposit_type_Non_Refund"]
target_test = X_test["deposit_type_Non_Refund"]
guesser = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=0).fit(X_train[no_deposit_cols], target_train)
guess = guesser.predict(X_test[no_deposit_cols])

print("accuracy of the guess:", round(accuracy_score(target_test, guess), 4))                                         # 0.9975
print("non-refundable bookings found:", int(((guess == 1) & (target_test == 1)).sum()), "of", int(target_test.sum()))   # 4741 of 4778

This is the lesson of the fairness audit of Week 7, in a different place: **deleting a column does not delete the information**.
Importance says which column the model *used*, not which information it *needs*. When two columns carry the same signal, the model picks one and the importance of the other looks small.

## Step 7. Is any column too good to be true?
The second column in the gain ranking is `required_car_parking_spaces`. Look at it in the data.

In [ ]:
parking = bookings["required_car_parking_spaces"] > 0
print(bookings.groupby(parking)["is_canceled"].agg(bookings="size", cancelled="sum", rate="mean").round(3))
# with a parking space: 7409 bookings, 0 cancelled. Without: rate 0.395

7,409 bookings asked for a parking space and **not one** was cancelled. Human behaviour is never that clean.
A more likely story: the parking space is entered in the system when the guest is at the front desk. If so, the column is known only for guests who arrived,
and it would not be available on the day we want to predict. This is **leakage**: information from the future inside the features.

The authors of the dataset warn that some fields change at check-in and differ between cancelled and other bookings for this reason.
We cannot prove it from the data. We can measure how much the model leans on the column.

In [ ]:
no_parking_cols = [c for c in X.columns if c != "required_car_parking_spaces"]
model_honest, auc_honest = test_auc(no_parking_cols)
print("without the parking column:", round(auc_honest, 3))   # 0.909

# And how good is this model on the bookings where the answer is not obvious?
others = X_test["deposit_type_Non_Refund"] == 0
proba_honest = model_honest.predict_proba(X_test[no_parking_cols])[:, 1]
print("the same model, only on bookings without a non-refundable deposit:", round(roc_auc_score(y_test[others], proba_honest[others]), 3))   # 0.864

Two readings of the headline AUC of 0.918:
- about one point of it comes from a column that is probably not available in time
- part of it comes from 4,778 test bookings that anybody could classify with one rule. On the remaining bookings, where the hotel needs help, the AUC without the parking column is 0.864

Explanations are a debugging tool. We found both issues by asking the model what it uses, not by looking at its accuracy.

## Step 8. Why this booking?
A waterfall explains one prediction: start from the average prediction, add the contribution of every column, arrive at the prediction for this booking.

In [ ]:
def explain(position):
    row = sample.iloc[[position]]
    print("prediction:", round(100 * model.predict_proba(row)[0, 1], 1), "% | cancelled in reality:", bool(y_test[row.index[0]]))
    shap.plots.waterfall(sv[position], max_display=9)

explain(96)   # 65.5 %: an online booking for the city hotel, 69 days ahead, no special request

In [ ]:
explain(7)    # 100.0 %: a group booking with a non-refundable deposit, 405 days ahead

In [ ]:
explain(1)    # 0.0 %: a booking with a parking space

The contributions always add up to the prediction. Check it for the first booking.

In [ ]:
i = 96
print(round(float(sv.base_values[i]), 1), "+", round(float(sv.values[i].sum()), 1), "=", round(float(sv.base_values[i] + sv.values[i].sum()), 1))   # 41.2 + 24.3 = 65.5

### What-if on one booking
Change one thing at a time on the first booking and ask the model again.

In [ ]:
booking = sample.iloc[[96]]
ask = lambda **changes: round(100 * model.predict_proba(booking.assign(**changes))[0, 1], 1)

print("as it is                    :", ask())                                                            # 65.5
print("one special request         :", ask(total_of_special_requests=1))                                 # 16.3
print("a parking space             :", ask(required_car_parking_spaces=1))                               # 0.0
print("a non-refundable deposit    :", ask(deposit_type_No_Deposit=0, deposit_type_Non_Refund=1))        # 100.0
print("booked 14 days ahead        :", ask(lead_time=14))                                                # 82.5
print("booked 3 days ahead         :", ask(lead_time=3))                                                 # 35.3

The model would bring this booking from 65.5% to 0.0% if we gave the guest a parking space, and to 100.0% if we asked for a deposit.
Neither is advice for the hotel. Both are the model repeating patterns of its data (Steps 5 and 7).

Note also the lead time: 69 days gives 65.5%, 14 days gives 82.5% and 3 days gives 35.3%. A model with trees of depth 11 is not smooth, and a counterfactual explanation can land on such a step.

## A closer look at one column: lead time
A dependence plot shows every booking of the sample: its lead time on the x axis, and what the lead time contributed to its prediction on the y axis.

In [ ]:
shap.plots.scatter(sv[:, "lead_time"])

bins = pd.cut(sample["lead_time"], [-1, 7, 30, 90, 180, 365, 800])
print(contrib["lead_time"].groupby(bins, observed=True).mean().round(1))   # from -13.5 points for bookings made in the last week to +8.2 for bookings made more than a year ahead

## Your turn
1. Add up the SHAP contributions of the eight `market_segment_...` columns, as we did for the deposit in Step 2. Where does the market segment rank now among the features?
2. Train the model without the deposit columns **and** without the parking column, and draw the beeswarm again. Which columns are at the top now? Is every direction one you can explain to a hotel manager?
3. `shap.TreeExplainer(model)` without `data` and `model_output` returns contributions in log-odds, the default of the library and the unit of most published SHAP plots. Draw the beeswarm in log-odds. Does the order of the columns change?
4. Train one model for the resort only (`hotel_Resort == 1`) and one for the city hotel only. Is the non-refundable deposit as important in both? Use Step 3 to explain what you see.
5. Set `RUN_SEARCH = True` in Step 1. How long does the search take on your machine, and how much AUC does it add to the fixed settings?